In [1]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 02 — DATA QUALITY & INTEGRITY ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CREDRESOLVE — DATA QUALITY & INTEGRITY ANALYSIS")
print("=" * 80)


# ------------------------------------------------------------
# 2. LOAD ALL RAW DATA
# ------------------------------------------------------------

csv_files = sorted(RAW_DIR.glob("*.csv"))

datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

print(f"Datasets loaded: {len(datasets)}")


# ------------------------------------------------------------
# 3. MISSING-VALUE ANALYSIS
# ------------------------------------------------------------

missing_records = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = int(df[column].isna().sum())

        missing_records.append({
            "dataset": name,
            "column": column,
            "rows": len(df),
            "missing_count": missing_count,
            "missing_pct": round(
                missing_count / len(df) * 100, 2
            )
        })

missing_df = pd.DataFrame(missing_records)

missing_df = (
    missing_df[
        missing_df["missing_count"] > 0
    ]
    .sort_values(
        ["missing_pct", "dataset"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

display(missing_df)

missing_df.to_csv(
    OUTPUT_DIR / "data_quality_missing_values.csv",
    index=False
)


# ------------------------------------------------------------
# 4. FULL-ROW DUPLICATE ANALYSIS
# ------------------------------------------------------------

duplicate_records = []

for name, df in datasets.items():

    duplicate_mask = df.duplicated(
        keep=False
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

    duplicate_records.append({
        "dataset": name,
        "rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_pct": round(
            duplicate_count / len(df) * 100, 2
        )
    })

duplicate_df = pd.DataFrame(
    duplicate_records
).sort_values(
    "duplicate_rows",
    ascending=False
)

print("\n" + "=" * 80)
print("FULL-ROW DUPLICATES")
print("=" * 80)

display(duplicate_df)

duplicate_df.to_csv(
    OUTPUT_DIR / "data_quality_duplicates.csv",
    index=False
)


# ------------------------------------------------------------
# 5. DUPLICATE IDENTIFIER ANALYSIS
# ------------------------------------------------------------
# We identify columns whose names suggest identifiers.
# We do NOT automatically delete duplicate IDs.
# ------------------------------------------------------------

identifier_keywords = [
    "id",
    "reference",
    "ref",
    "code"
]

identifier_records = []

for name, df in datasets.items():

    for column in df.columns:

        column_lower = column.lower()

        if any(
            keyword in column_lower
            for keyword in identifier_keywords
        ):

            non_null = df[column].dropna()

            identifier_records.append({
                "dataset": name,
                "column": column,
                "rows": len(df),
                "missing_count": int(
                    df[column].isna().sum()
                ),
                "unique_values": int(
                    df[column].nunique(
                        dropna=True
                    )
                ),
                "duplicate_non_null_values": int(
                    non_null.duplicated().sum()
                )
            })

identifier_df = pd.DataFrame(
    identifier_records
)

print("\n" + "=" * 80)
print("IDENTIFIER QUALITY")
print("=" * 80)

display(identifier_df)

identifier_df.to_csv(
    OUTPUT_DIR / "data_quality_identifiers.csv",
    index=False
)


# ------------------------------------------------------------
# 6. CANDIDATE PRIMARY-KEY VALIDATION
# ------------------------------------------------------------
# A candidate key should have:
#   - no nulls
#   - unique values
#
# This is still an initial validation.
# Business relationships will be checked separately.
# ------------------------------------------------------------

key_records = []

for name, df in datasets.items():

    for column in df.columns:

        null_count = int(
            df[column].isna().sum()
        )

        unique_count = int(
            df[column].nunique(
                dropna=True
            )
        )

        key_records.append({
            "dataset": name,
            "column": column,
            "rows": len(df),
            "null_count": null_count,
            "unique_count": unique_count,
            "is_unique": unique_count == len(df),
            "has_nulls": null_count > 0
        })

key_df = pd.DataFrame(key_records)

candidate_key_df = key_df[
    (key_df["is_unique"]) &
    (~key_df["has_nulls"])
].copy()

print("\n" + "=" * 80)
print("VALID CANDIDATE KEYS")
print("=" * 80)

display(candidate_key_df)

candidate_key_df.to_csv(
    OUTPUT_DIR / "validated_candidate_keys.csv",
    index=False
)


# ------------------------------------------------------------
# 7. DATA TYPE / FORMAT CHECK
# ------------------------------------------------------------

type_records = []

for name, df in datasets.items():

    for column in df.columns:

        type_records.append({
            "dataset": name,
            "column": column,
            "dtype": str(df[column].dtype),
            "sample_non_null_value": (
                df[column].dropna().iloc[0]
                if df[column].notna().any()
                else None
            )
        })

type_df = pd.DataFrame(
    type_records
)

print("\n" + "=" * 80)
print("DATA TYPE PROFILE")
print("=" * 80)

display(type_df.head(50))

type_df.to_csv(
    OUTPUT_DIR / "data_quality_data_types.csv",
    index=False
)


# ------------------------------------------------------------
# 8. DATE / TIMESTAMP QUALITY CHECK
# ------------------------------------------------------------

date_keywords = [
    "date",
    "time",
    "timestamp",
    "created",
    "updated",
    "scheduled",
    "started",
    "ended",
    "occurred",
    "paid",
    "opened",
    "closed",
    "effective"
]

timestamp_records = []

for name, df in datasets.items():

    for column in df.columns:

        if any(
            keyword in column.lower()
            for keyword in date_keywords
        ):

            original = df[column]

            parsed = pd.to_datetime(
                original,
                errors="coerce"
            )

            invalid_count = int(
                (
                    original.notna()
                    & parsed.isna()
                ).sum()
            )

            timestamp_records.append({
                "dataset": name,
                "column": column,
                "original_dtype": str(
                    original.dtype
                ),
                "non_null_values": int(
                    original.notna().sum()
                ),
                "invalid_dates": invalid_count,
                "min_date": (
                    parsed.min()
                    if parsed.notna().any()
                    else None
                ),
                "max_date": (
                    parsed.max()
                    if parsed.notna().any()
                    else None
                )
            })

timestamp_df = pd.DataFrame(
    timestamp_records
)

print("\n" + "=" * 80)
print("TIMESTAMP QUALITY")
print("=" * 80)

display(timestamp_df)

timestamp_df.to_csv(
    OUTPUT_DIR / "data_quality_timestamps.csv",
    index=False
)


# ------------------------------------------------------------
# 9. SUSPICIOUS NEGATIVE NUMERIC VALUES
# ------------------------------------------------------------
# We flag negative values for investigation.
# We do NOT assume that every negative number is wrong.
# ------------------------------------------------------------

negative_records = []

for name, df in datasets.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:

        negative_count = int(
            (df[column] < 0).sum()
        )

        if negative_count > 0:

            negative_records.append({
                "dataset": name,
                "column": column,
                "negative_count": negative_count,
                "negative_pct": round(
                    negative_count / len(df) * 100,
                    2
                ),
                "minimum_value": df[column].min()
            })

negative_df = pd.DataFrame(
    negative_records
)

print("\n" + "=" * 80)
print("NEGATIVE NUMERIC VALUES — INVESTIGATION FLAGS")
print("=" * 80)

display(negative_df)

negative_df.to_csv(
    OUTPUT_DIR / "data_quality_negative_values.csv",
    index=False
)


# ------------------------------------------------------------
# 10. CONSTANT / LOW-VARIATION COLUMNS
# ------------------------------------------------------------

variation_records = []

for name, df in datasets.items():

    for column in df.columns:

        unique_count = df[column].nunique(
            dropna=True
        )

        variation_records.append({
            "dataset": name,
            "column": column,
            "unique_values": int(unique_count),
            "rows": len(df),
            "unique_pct": round(
                unique_count / len(df) * 100,
                2
            )
        })

variation_df = pd.DataFrame(
    variation_records
)

low_variation_df = (
    variation_df[
        variation_df["unique_values"] <= 5
    ]
    .sort_values(
        ["dataset", "unique_values"]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("LOW-VARIATION COLUMNS — INVESTIGATION FLAGS")
print("=" * 80)

display(low_variation_df)

low_variation_df.to_csv(
    OUTPUT_DIR / "data_quality_low_variation.csv",
    index=False
)


# ------------------------------------------------------------
# 11. DATA QUALITY SUMMARY
# ------------------------------------------------------------

quality_summary = pd.DataFrame([{

    "datasets_analyzed":
        len(datasets),

    "datasets_with_missing_values":
        int(
            missing_df["dataset"].nunique()
        ) if not missing_df.empty else 0,

    "columns_with_missing_values":
        len(missing_df),

    "datasets_with_duplicate_rows":
        int(
            (duplicate_df["duplicate_rows"] > 0).sum()
        ),

    "total_duplicate_rows":
        int(
            duplicate_df["duplicate_rows"].sum()
        ),

    "identifier_columns_reviewed":
        len(identifier_df),

    "candidate_keys_identified":
        len(candidate_key_df),

    "timestamp_columns_reviewed":
        len(timestamp_df),

    "timestamp_columns_with_invalid_dates":
        int(
            (timestamp_df["invalid_dates"] > 0).sum()
        ) if not timestamp_df.empty else 0,

    "numeric_columns_with_negative_values":
        len(negative_df),

    "low_variation_columns_flagged":
        len(low_variation_df)
}])


print("\n" + "=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

display(quality_summary)

quality_summary.to_csv(
    OUTPUT_DIR / "data_quality_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 12. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA QUALITY ANALYSIS COMPLETE")
print("=" * 80)

print("Raw source files were NOT modified.")
print(f"Quality outputs saved to: {OUTPUT_DIR}")

CREDRESOLVE — DATA QUALITY & INTEGRITY ANALYSIS
Datasets loaded: 18

MISSING VALUES


,dataset,column,rows,missing_count,missing_pct
0,borrowers,email,30600,895,2.92
1,borrowers,phone,30600,614,2.01
2,call_attempts,vendor_id,120000,2400,2.00
3,calls,agent_id,91350,1827,2.00
4,accounts,borrower_id,30000,455,1.52
5,payments,payment_reference,25500,382,1.50
6,field_visits,scheduled_at,25000,250,1.00



FULL-ROW DUPLICATES


,dataset,rows,duplicate_rows,duplicate_pct
7,calls,91350,2542,2.78
4,borrowers,30600,1200,3.92
17,whatsapp_events,60600,1200,1.98
13,payments,25500,972,3.81
1,accounts,30000,0,0.00
0,account_status_history,60000,0,0.00
5,call_attempts,120000,0,0.00
6,call_dispositions,35000,0,0.00
3,agents,30000,0,0.00
2,agent_sessions,15000,0,0.00



IDENTIFIER QUALITY


...


LOW-VARIATION COLUMNS — INVESTIGATION FLAGS


,dataset,column,unique_values,rows,unique_pct
0,account_status_history,source,5,60000,0.01
1,accounts,timezone,3,30000,0.01
2,accounts,schema_version,3,30000,0.01
3,accounts,risk_segment,4,30000,0.01
4,accounts,status,4,30000,0.01
5,accounts,loan_type,5,30000,0.02
6,agent_sessions,timezone,2,15000,0.01
7,agent_sessions,channel,4,15000,0.03
8,agents,status,3,30000,0.01
9,agents,team,5,30000,0.02



DATA QUALITY SUMMARY


,datasets_analyzed,datasets_with_missing_values,columns_with_missing_values,datasets_with_duplicate_rows,total_duplicate_rows,identifier_columns_reviewed,candidate_keys_identified,timestamp_columns_reviewed,timestamp_columns_with_invalid_dates,numeric_columns_with_negative_values,low_variation_columns_flagged
0,18,6,7,4,5914,64,17,12,5,0,37



DATA QUALITY ANALYSIS COMPLETE
Raw source files were NOT modified.
Quality outputs saved to: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
